# Aula 4 — Validação na bancada e anti-windup

**Projeto final · Laboratório de Controle Automático · Ifes Guarapari · 03/12 · 1h40**

**Objetivo da aula:** provar, com ensaios repetidos, que o controlador atende às especificações
1–3 na bancada; provocar a **saturação** do PWM e mostrar que o **anti-windup** cumpre a
especificação 4; comparar o que foi medido com o que o modelo previa. **Fecha o Trabalho 2.**

**Papéis na bancada (troquem a cada aula):** 🔧 **operador(a)** — mexe no hardware e grava o
firmware · 💻 **programador(a)** — roda e edita o notebook · ✍️ **relator(a)** — escreve as
respostas ❓ e confere os números. Com 2 pessoas, o relator acumula com o programador.

**Legenda:** ⏱️ tempo sugerido · 📘 teoria (leia antes de fazer) · 🔧 bancada · 💻 notebook ·
✏️ complete o código · ❓ responda · ✅ checkpoint — **chame o professor**.

| ⏱️ | Parte | O que se faz |
|---|---|---|
| 0–35 min | 1 | Três ensaios nominais: especificações 1–3 |
| 35–70 min | 2 | Saturação e anti-windup: especificação 4 |
| 70–90 min | 3 | Medido × simulado |
| 90–100 min | ✅ | Checkpoint 4 (fim do Trabalho 2) |

In [ ]:
# ⚙️ CONFIGURAÇÃO — rode esta célula sempre que abrir o notebook
import numpy as np
import matplotlib.pyplot as plt
import control as ct
from scipy.optimize import minimize
from ferramentas_projeto import *

PORTA = "COM5"     # ✏️ porta do Arduino (Arduino IDE > Ferramentas > Porta). Linux: "/dev/ttyUSB0" ou "/dev/ttyACM0"
BANCADA = "1"      # ✏️ número da bancada

s = ct.tf("s")
Ts = 0.02          # período de amostragem do firmware [s]
plt.rcParams.update({"axes.grid": True, "grid.alpha": 0.3})
P = carregar_parametros()          # resultados das aulas anteriores (dados/parametros_bancada.json)
print("bancada", BANCADA, "| parâmetros já salvos:", list(P))

In [ ]:
GEMEO = P["GEMEO"]
CFG = dict(P["CFG_FINAL"])
CFG["CPG"] = P["CPG"]
print("controlador final:", CFG)

## Parte 1 — Três ensaios nominais · ⏱️ 35 min

O atrito muda um pouco de um ensaio para outro, então um ensaio só não prova nada. Repetimos o
perfil completo **três vezes** e declaramos a especificação atendida pelo **pior** dos três (só
o degrau de 90° vale para as especificações; os outros dois vão para a discussão).

**🔧💻 Passo 1.1** — três ensaios (17 s cada).

In [ ]:
nominais = [coletar(f"mf_nominal_{i}", PORTA, **CFG) for i in (1, 2, 3)]
ms = [metricas_ensaio(d) for d in nominais]
for i, m in enumerate(ms, 1):
    imprime_metricas(m, f"\nBANCADA — repetição {i}")
sim = simular(GEMEO, **CFG)
imprime_metricas(metricas_ensaio(sim), "\nGÊMEO DIGITAL — previsão")

In [ ]:
deg90 = [m[0] for m in ms]                    # o 1º degrau do perfil é o 0° → 90°
pior = dict(Mp=max(m["Mp"] for m in deg90), ts=max(m["ts"] for m in deg90),
            ess=max((m["ess"] for m in deg90), key=abs))
print("Degrau de 90°, pior das 3 repetições:")
veredito(pior)
plota_ensaio(sim, nominais[0], rotulos=["gêmeo", "bancada (rep. 1)"], titulo="Ensaio nominal")
plt.show()

## Parte 2 — Saturação e anti-windup · ⏱️ 35 min

📘 **Saturação e *windup*.** O PWM não passa de ±255 (aqui vamos **limitar** a ±80 de propósito,
para forçar a saturação no degrau de 90°). Enquanto o PWM está no teto, o controlador **pede** mais
do que recebe e o erro demora a cair — e o **integrador continua somando** esse erro grande. Quando
o eixo chega à referência, o integrador está "cheio" (*wind-up*, como uma mola enrolada): ele
mantém o PWM alto e o eixo passa da referência. Com atrito, o eixo ainda pode **ficar preso** do
outro lado até o integrador "esvaziar".

📘 **Anti-windup por *clamping*.** A cada amostra, se o PWM está saturado **e** o erro empurraria
ainda mais para a saturação, o firmware **não integra** naquela amostra. O integrador fica
"congelado" enquanto não pode ajudar. No firmware: `AW=1` liga, `AW=0` desliga.

📘 **Como enxergar isso.** A 5ª coluna da telemetria, `u_pid`, é o que o PID **pediu** (antes do
limite); `pwm` é o que foi **aplicado**. Durante a saturação, `u_pid` > `pwm`.

**Protocolo da especificação 4:** mesmo controlador, `PWM_MAX = 80`, com e sem anti-windup.
Atende se $M_p$(saturado, AW ligado) ≤ $M_p$(nominal) + 5 pontos percentuais.

**🔧💻 Passo 2.1** — confira o limite e rode os dois ensaios.

In [ ]:
PWM_SAT = 80
if PWM_SAT < 1.5 * max(P["Fs_pos"], P["Fs_neg"]):
    print("⚠️ PWM_SAT muito perto do atrito de partida — o motor mal vai andar. Chame o professor.")
sat = {}
for aw in (1, 0):
    sat[aw] = coletar(f"mf_sat_aw{aw}", PORTA, **dict(CFG, PWM_MAX=PWM_SAT, AW=aw))
    imprime_metricas(metricas_ensaio(sat[aw], pwm_max=PWM_SAT),
                     f"\nBANCADA — PWM_MAX = {PWM_SAT}, anti-windup {'LIGADO' if aw else 'DESLIGADO'}")

In [ ]:
Mp_nom = deg90[0]["Mp"]
dMp = {aw: metricas_ensaio(sat[aw], pwm_max=PWM_SAT)[0]["Mp"] - Mp_nom for aw in (1, 0)}
print(f"Degrau de 90°: Mp nominal = {Mp_nom:.1f} %")
print(f"   AW ligado   : ΔMp = {dMp[1]:+.1f} p.p.  ->  {'✅ atende' if dMp[1] <= 5 else '❌ não atende'} a especificação 4")
print(f"   AW desligado: ΔMp = {dMp[0]:+.1f} p.p.")

fig, ax = plt.subplots(2, 1, figsize=(9, 6), sharex=True)
for aw, cor, nome in [(0, "tab:red", "AW desligado"), (1, "tab:blue", "AW ligado")]:
    d = sat[aw]
    ax[0].plot(d["t"], d["angle"], color=cor, lw=1.7, label=nome)
    ax[1].plot(d["t"], d["u_pid"], color=cor, ls=":", lw=1.4, label=f"u pedido — {nome}")
    ax[1].plot(d["t"], d["pwm"], color=cor, lw=1.6, label=f"PWM aplicado — {nome}")
ax[0].axhline(90, color="k", ls="--", lw=0.8); ax[0].set_ylim(-5, 110); ax[0].legend()
ax[0].set_ylabel("ângulo (graus)"); ax[0].set_title(f"Especificação 4 — PWM limitado a ±{PWM_SAT} (zoom no degrau de 90°)")
ax[1].axhline(PWM_SAT, color="0.5", ls="--", lw=0.8); ax[1].set_ylim(-60, 250)
ax[1].set_xlim(1.9, 4.5); ax[1].set_xlabel("tempo (s)"); ax[1].set_ylabel("PWM"); ax[1].legend(fontsize=7)
plt.show()

❓ **2.1** No gráfico de baixo: durante a saturação, o PWM aplicado é igual nos dois casos. Então
onde está a diferença? Compare as curvas pontilhadas (`u_pid`) logo depois que o eixo chega perto
de 90°, e explique o sobressinal do caso sem anti-windup.

✍️ **Resposta da bancada:** *(escreva aqui)*

## Parte 3 — Medido × simulado · ⏱️ 20 min

Respondam olhando as tabelas e os gráficos desta aula e das anteriores.

❓ **3.1** O sobressinal **medido** é maior ou menor que o do **modelo linear** (Aula 3, Parte 1)?
Que fenômeno da bancada, ausente do modelo linear, explica a diferença?

✍️ **Resposta da bancada:** *(escreva aqui)*

❓ **3.2** Os três degraus do perfil se comportam igual? Compare o 0° → 90° com o 90° → −90°
(180°!). Que medições das Aulas 2 e 3 explicam as diferenças?

✍️ **Resposta da bancada:** *(escreva aqui)*

❓ **3.3** As três repetições deram exatamente os mesmos números? Por que se declara a
especificação pelo pior caso?

✍️ **Resposta da bancada:** *(escreva aqui)*

## ✅ Checkpoint 4 — fim do Trabalho 2 · ⏱️ 10 min

Rode, salve o notebook e chame o professor.

In [ ]:
P = salvar_parametros(pior_nominal=pior, dMp_aw1=dMp[1], dMp_aw0=dMp[0])
print(f"Bancada {BANCADA} — Trabalho 2 (Aulas 3 e 4)")
print(f"  controlador: kp = {CFG['KP']}, ki = {CFG['KI']}, kd = {CFG['KD']}, ZM = {CFG['ZM']}, EM = {CFG['EM']}")
print("  degrau de 90° (pior de 3):")
veredito(pior)
print(("✅" if dMp[1] <= 5 else "❌") + f" 4. ΔMp com anti-windup = {dMp[1]:+.1f} p.p. ≤ 5 p.p. (sem: {dMp[0]:+.1f} p.p.)")